In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# Understanding Warps in CUDA


> **Overview**
>
> * **Time:** 30 min
> 
> 1. Learn about warps in CUDA programming.
> 1. Understand how warps affect performance and execution in CUDA kernels.

Warps are a fundamental concept in CUDA programming, representing how threads are organized and executed on NVIDIA GPUs. Understanding warps is crucial
for writing efficient CUDA kernels.

## What is a Warp?

A warp is a group of threads that execute the same instruction at the same time on a CUDA-enabled GPU.

* A **warp** consists of **32 threads** (in all current GPUs) that are executed **in parallel**.
* When a CUDA kernel is launched, threads are grouped into **blocks**, which are further divided into **warps** of 32 threads each.
* The 32 threads in a warp execute the **same instruction simultaneously** (SIMT - Single Instruction, Multiple Threads), but on **different data**.

## Why Use Warps?


Warps are designed to:

* Maximize **instruction-level parallelism**.
* Efficiently utilize the GPU's **SIMD-style execution units**.
* Enable lightweight and efficient scheduling using the **warp scheduler**.

## Thread Grouping into Warps


If a thread block contains 128 threads:

```text
128 threads / 32 threads per warp = 4 warps
```

* Threads 0–31 form warp 0.
* Threads 32–63 form warp 1.
* Threads 64–95 form warp 2.
* Threads 96–127 form warp 3.

## Execution Behavior


* **Threads in a warp execute in lockstep**, following the same instruction.
* If threads diverge in control flow (e.g., due to `if` conditions), **warp divergence** occurs, causing **serialized execution** and reduced performance.

## Warp Divergence


Warp divergence happens when threads in the same warp **follow different execution paths**.

Example:

```cpp
int tid = threadIdx.x;
if (tid % 2 == 0) {
    // Even threads do this
} else {
    // Odd threads do that
}
```

In this example, even and odd threads follow different branches, leading to warp divergence. This means that the warp will execute the even threads first,
then the odd threads, effectively serializing the execution and reducing performance.


Similarly it is a bad practice to have threads numbers that are not a multiple of 32 in a warp. This can lead to inefficient use of resources and increased
execution time.

For example, if a CUDA thread block has 30 threads, the last two threads in the warp will not be utilized, leading to wasted resources.
If a CUDA thread block has 40 threads, the block will be split as follows:

* Warp 0: threads 0–31
* Warp 1: threads 32–39 (only 8 threads)

So, the block will contain 2 warps, but the second warp is partially full (only 8 active threads).

* Warp 0 will execute normally with all 32 threads.
* Warp 1 will execute with only 8 active threads and the remaining 24 threads inactive (masked out).

The GPU still treats it as a full warp in terms of scheduling. Performance-wise, you're paying the cost of a full warp but using only part of it.


> **Important**
>
> * Write control flow to **minimize warp divergence**.
> * Ensure that threads access **contiguous memory** for coalesced memory access.
> * Aim for all threads in a warp to **follow the same path** for maximum efficiency.

## Some additional details


* Each SM is divided into processing partitions (also called sub-partitions).

* Each partition contains its own warp scheduler and instruction dispatch unit, along with a portion of the execution pipelines.

* On Volta (V100), an SM has 4 processing partitions, each with a warp scheduler, for a total of 4 schedulers per SM.

* An SM can host up to 64 resident warps (2048 threads).

* In each cycle, up to 4 warps can issue instructions (one per scheduler), while the rest wait.

* This deep pool of resident warps allows the GPU to quickly swap in ready warps when others are stalled (e.g., on memory or synchronization), ensuring high utilization and latency hiding.

In [ ]:
# Processing Partitions
display(Image(filename=str(repo_root / "docs/source/tutorial/figs/sm-partitions.png")))

> **Key Points**
>
> 1. A warp is a group of 32 threads that execute the same instruction simultaneously.
> 1. Warps are fundamental to CUDA's execution model, enabling parallel processing.
> 1. Warp divergence can lead to performance issues; aim to minimize it.
> 1. Threads should be organized to maximize coalesced memory access and minimize divergence.
> 1. Understanding warps is essential for writing efficient CUDA kernels and optimizing performance.

## Exercise 05: Warps

**Time: 20 min**

The job records timestamps for one block of 256 threads (eight warps).
[Exercise instructions](../docs/source/tutorial/exercises/05-warps.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

<a href="../exercises/05-warps/warp-timing.cu" target="_blank" rel="noopener noreferrer">Open warp-timing.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 05-warps.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


In [ ]:
!qstat -u "{username}"
show_job_output("05-warps", job_id)


### Plot the completed job's timestamps

After the job finishes, this cell reads `timestamps.csv` from its exercise folder.
The plots use standard Python and inline SVG; no plotting packages are needed. Timestamps
are measurement points, not a guarantee of warp scheduling order.

In [ ]:
import csv

output_dir = repo_root / "exercises/05-warps"
if outputs_ready([output_dir / "timestamps.csv"], submitted_at):
    with (output_dir / "timestamps.csv").open(newline="") as stream:
        data = [{key: int(value) for key, value in row.items()} for row in csv.DictReader(stream)]
    assert len(data) == 256
    assert [row["thread"] for row in data] == list(range(256))
    assert all(row["warp"] == row["thread"] // 32 and row["lane"] == row["thread"] % 32
               for row in data)
    origin = min(row["clock"] for row in data)
    spread = max(row["clock"] for row in data) - origin
    palette = ["#dbeafe", "#ffedd5", "#dcfce7", "#f3e8ff", "#fce7f3", "#cffafe", "#e2e8f0", "#fef3c7"]
    parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="1280" height="300" viewBox="0 0 1280 300">',
             '<rect width="1280" height="300" fill="white"/>',
             '<g font-family="monospace" fill="#172033">']
    for panel, title in enumerate(["Warp membership", "Timestamp offset (SM clock cycles)"]):
        x0 = 48 + panel * 640
        parts.append(f'<text x="{x0}" y="25" font-size="16">{title}</text>')
        for row in data:
            x = x0 + row["lane"] * 18
            y = 45 + row["warp"] * 24
            if panel == 0:
                color = palette[row["warp"]]
            else:
                ratio = (row["clock"] - origin) / max(1, spread)
                red, green, blue = (round(255 - delta * ratio) for delta in [218, 156, 20])
                color = f"#{red:02x}{green:02x}{blue:02x}"
            parts.append(f'<rect x="{x}" y="{y}" width="18" height="24" fill="{color}" stroke="#94a3b8" stroke-width="0.5"/>')
        for warp in range(8):
            parts.append(f'<text x="{x0 - 24}" y="{62 + warp * 24}" font-size="12">{warp}</text>')
        for lane in [0, 7, 15, 23, 31]:
            parts.append(f'<text x="{x0 + lane * 18}" y="255" font-size="12">{lane}</text>')
        parts.append(f'<text x="{x0}" y="280" font-size="12">Rows: warp ID; columns: lane ID</text>')
    parts.append('</g></svg>')
    svg = "".join(parts)
    plot_dir = repo_root / "notebook-output/05-warps"
    plot_dir.mkdir(parents=True, exist_ok=True)
    (plot_dir / "warp-timing.svg").write_text(svg)
    display(SVG(data=svg))
    print(f"Timestamp offsets: 0 to {spread} SM clock cycles (white to blue).")
    print("Warp   earliest   latest   (SM clock cycles after first reading)")
    for warp in range(8):
        offsets = [row["clock"] - origin for row in data if row["warp"] == warp]
        print(f"{warp:4d}   {min(offsets):8d}   {max(offsets):6d}")


Repeat the job and compare the timestamp patterns. Does warp 0 always record
first? Why can threads from different warps have similar timestamps?